In [2]:
import requests 

In [ ]:
london_regions = {
    "Central London" : {
        "court_name" : "The Regent's Park", 
        "lat" : 51.528336, 
        "long" : -0.154449
    },
    "North London" : {
        "court_name" : "Finsbury Park", 
        "lat" : 51.571402, 
        "long" : -0.103208
    },
    "South London" : {
        "court_name" : "Clapham Common", 
        "lat" : 51.454515, 
        "long" : -0.152749
    }, 
    "East London" : {
        "court_name" : "Stratford Park",
        "lat" : 51.537542, 
        "long" : 0.00564
    }, 
    "West London" : {
         "court_name" : "Battersea Park", 
        "lat" : 51.480182, 
        "long" : -0.155703
    }
}

In [4]:
import openmeteo_requests

In [5]:
import pandas as pd
import requests_cache
from retry_requests import retry

In [6]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


In [7]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 51.537542,
	"longitude": 0.00564,
	"daily": ["sunrise", "sunset"],
	"hourly": ["temperature_2m", "apparent_temperature", "precipitation_probability", "rain", "weather_code", "wind_speed_10m", "wind_gusts_10m"],
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params=params)

In [8]:
response = responses[0]


In [9]:
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 51.540000915527344°N -2.384185791015625e-07°E
Elevation: 10.0 m asl
Timezone: b'Europe/London'None
Timezone difference to GMT+0: 0s


In [10]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(2).ValuesAsNumpy()
hourly_rain = hourly.Variables(3).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(6).ValuesAsNumpy()

In [11]:
hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}


In [12]:
hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["rain"] = hourly_rain
hourly_data["weather_code"] = hourly_weather_code
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m

In [13]:
hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe

,date,temperature_2m,apparent_temperature,precipitation_probability,rain,weather_code,wind_speed_10m,wind_gusts_10m
0,2026-01-31 00:00:00+00:00,8.4825,6.319663,0.0,0.0,3.0,8.404284,18.719999
1,2026-01-31 01:00:00+00:00,8.4325,6.095923,0.0,0.0,2.0,9.255571,20.160000
2,2026-01-31 02:00:00+00:00,8.2325,5.616308,0.0,0.0,2.0,9.832680,23.400000
3,2026-01-31 03:00:00+00:00,7.8825,5.642735,0.0,0.0,2.0,7.771331,20.880001
4,2026-01-31 04:00:00+00:00,7.5325,5.428052,0.0,0.0,2.0,6.830519,16.919998
...,...,...,...,...,...,...,...,...
163,2026-02-06 19:00:00+00:00,4.4430,0.561682,26.0,0.1,61.0,14.471821,33.480000
164,2026-02-06 20:00:00+00:00,4.4930,0.701875,24.0,0.1,61.0,14.113653,32.399998
165,2026-02-06 21:00:00+00:00,4.4930,0.864835,23.0,0.1,61.0,13.397612,31.319998
166,2026-02-06 22:00:00+00:00,4.4930,0.946291,22.0,0.7,61.0,13.039754,30.960001


In [14]:
# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_sunrise = daily.Variables(0).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(1).ValuesInt64AsNumpy()

In [15]:
daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

In [16]:
daily_data["sunrise"] = daily_sunrise
daily_data["sunset"] = daily_sunset

daily_dataframe = pd.DataFrame(data = daily_data)

In [22]:
daily_dataframe

,date,sunrise,sunset
0,2026-01-31 00:00:00+00:00,1769845176,1769878024
1,2026-02-01 00:00:00+00:00,1769931483,1769964533
2,2026-02-02 00:00:00+00:00,1770017788,1770051043
3,2026-02-03 00:00:00+00:00,1770104091,1770137554
4,2026-02-04 00:00:00+00:00,1770190393,1770224064
5,2026-02-05 00:00:00+00:00,1770276693,1770310574
6,2026-02-06 00:00:00+00:00,1770362992,1770397085


In [23]:
hourly_dataframe.head(25)

,date,temperature_2m,apparent_temperature,precipitation_probability,rain,weather_code,wind_speed_10m,wind_gusts_10m
0,2026-01-31 00:00:00+00:00,8.482500,6.319663,0.0,0.0,3.0,8.404284,18.719999
1,2026-01-31 01:00:00+00:00,8.432500,6.095923,0.0,0.0,2.0,9.255571,20.160000
2,2026-01-31 02:00:00+00:00,8.232500,5.616308,0.0,0.0,2.0,9.832680,23.400000
3,2026-01-31 03:00:00+00:00,7.882500,5.642735,0.0,0.0,2.0,7.771331,20.880001
4,2026-01-31 04:00:00+00:00,7.532500,5.428052,0.0,0.0,2.0,6.830519,16.919998
5,2026-01-31 05:00:00+00:00,7.182500,5.153539,0.0,0.0,2.0,6.287130,14.040000
6,2026-01-31 06:00:00+00:00,6.782500,4.945966,0.0,0.0,1.0,5.091168,13.320000
7,2026-01-31 07:00:00+00:00,6.682500,4.777571,3.0,0.0,3.0,5.411986,11.879999
8,2026-01-31 08:00:00+00:00,6.832500,5.048844,8.0,0.0,80.0,4.802999,11.159999
9,2026-01-31 09:00:00+00:00,7.282500,5.482487,3.0,0.0,2.0,5.351785,10.080000


In [21]:
hourly_dataframe.iloc[24:48]

,date,temperature_2m,apparent_temperature,precipitation_probability,rain,weather_code,wind_speed_10m,wind_gusts_10m
24,2026-02-01 00:00:00+00:00,7.932500,5.964530,0.0,0.0,3.0,6.489992,14.400000
25,2026-02-01 01:00:00+00:00,7.882500,6.097754,0.0,0.0,80.0,5.411986,14.040000
26,2026-02-01 02:00:00+00:00,7.882500,6.124331,0.0,0.0,3.0,5.483356,13.679999
27,2026-02-01 03:00:00+00:00,7.632500,5.694995,0.0,0.0,3.0,6.830519,14.759999
28,2026-02-01 04:00:00+00:00,7.382500,5.494737,0.0,0.0,3.0,5.860375,14.400000
29,2026-02-01 05:00:00+00:00,7.082500,5.378184,3.0,0.0,3.0,4.394360,12.599999
30,2026-02-01 06:00:00+00:00,6.882500,5.274772,5.0,0.0,3.0,3.671294,9.360000
31,2026-02-01 07:00:00+00:00,6.382500,4.916261,0.0,0.0,3.0,2.880000,7.559999
32,2026-02-01 08:00:00+00:00,6.032500,4.623044,0.0,0.0,3.0,2.189795,6.120000
33,2026-02-01 09:00:00+00:00,6.382500,4.882889,0.0,0.0,3.0,2.880000,6.120000
